# Milestone M3–M4: Preprocessing Sentral & Stratified Partitioning
## Kelompok 4 — CNN for Text Classification (Indonesian Hate Speech Detection)

**Mata Kuliah:** Workshop Proyek Sistem Cerdas 2026  
**Dosen Pengampu:** Dr. Selvia Ferdiana Kusuma, M.Kom  
**Dataset:** `data/raw/indotoxic2024_annotated_data_v2_final.csv` (28.448 baris)  

### Pipeline Preprocessing Sentral:
1. **Pembersihan Teks Mentah (`TextCleaner`)**: Hapus URLs, mention, hashtag, tanda baca berlebih, dan normalisasi lowercase.
2. **Filtering Spam/Noise**: Mengeliminasi baris dengan flag `is_noise_or_spam_text` atau teks kosong pasca pembersihan.
3. **Stratified Splitting (`DataSplitter`)**: Membagi data ke Train (70%), Validation (15%), dan Test (15%) dengan menjaga rasio target.
4. **Penyimpanan Splits**: Menyimpan artefak ke `data/splits/train.csv`, `val.csv`, dan `test.csv`.
5. **Vocabulary Fitting & Tokenisasi (`TextTokenizer`)**: Fitting vocabulary HANYA pada data `train` untuk mencegah *data leakage*.
6. **Padding Sekuens (`SequencePadder`)**: Padding dan truncating ke dimensi `MAX_LEN = 128`.

In [ ]:
import sys
from pathlib import Path
import os
import ast

# Set root path project
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np

from src.utils.config import Config
from src.utils.seed import set_seed
from src.preprocessing.cleaner import TextCleaner
from src.preprocessing.splitter import DataSplitter
from src.preprocessing.tokenizer import TextTokenizer
from src.preprocessing.padder import SequencePadder

set_seed(Config.SEED)
print(f"Project root : {ROOT_DIR}")
print(f"Config Model : VOCAB_SIZE={Config.VOCAB_SIZE}, MAX_LEN={Config.MAX_LEN}")

### 1. Pemuatan Dataset Mentah & Ekstraksi Label Konsensus

In [ ]:
raw_csv_path = ROOT_DIR / Config.RAW_DATA_CSV
df_raw = pd.read_csv(raw_csv_path)
print(f"Dataset mentah dimuat: {len(df_raw):,} baris")

def aggregate_label(val):
    if isinstance(val, str):
        try:
            arr = ast.literal_eval(val)
            nums = [int(x) for x in arr]
            return 1 if (sum(nums) / len(nums)) >= 0.5 else 0
        except Exception:
            return 0
    return int(val) if not pd.isna(val) else 0

df_raw["label"] = df_raw["toxicity"].apply(aggregate_label)
print("Distribusi label awal:")
print(df_raw["label"].value_counts())

### 2. Text Cleaning & Noise Filtering dengan `TextCleaner`

In [ ]:
cleaner = TextCleaner(
    remove_urls=True,
    remove_mentions=True,
    remove_hashtags=True,
    remove_punctuation=True,
    lowercase=True
)

# Jalankan batch cleaning
print("Menjalankan text cleaning...")
df_raw["text_clean"] = cleaner.clean_batch(df_raw["text"].astype(str).tolist())

# Filter baris yang kosong atau spam jika ditandai anotator
df_filtered = df_raw[df_raw["text_clean"].str.len() > 2].copy()
print(f"Data tersisa setelah filter teks kosong: {len(df_filtered):,} baris")

# Simpan cleaned dataset
processed_out = ROOT_DIR / Config.PROCESSED_DATA_PATH
os.makedirs(processed_out.parent, exist_ok=True)
cols_to_save = ["text_id", "text_clean", "topic", "label"]
df_filtered[cols_to_save].to_csv(processed_out, index=False)
print(f"Cleaned dataset tersimpan di: {processed_out}")
df_filtered[cols_to_save].head(3)

### 3. Stratified Data Splitting (70% Train, 15% Val, 15% Test) dengan `DataSplitter`

In [ ]:
splitter = DataSplitter(
    train_ratio=Config.TRAIN_RATIO,
    val_ratio=Config.VAL_RATIO,
    test_ratio=Config.TEST_RATIO,
    seed=Config.SEED
)

train_df, val_df, test_df = splitter.split(df_filtered, stratify_col="label")
saved_paths = splitter.save_splits(
    train_df[cols_to_save],
    val_df[cols_to_save],
    test_df[cols_to_save],
    output_dir=str(ROOT_DIR / Config.SPLITS_DIR)
)

print(f"Train set : {len(train_df):,} baris (Toxic: {train_df['label'].mean()*100:.2f}%)")
print(f"Val set   : {len(val_df):,} baris (Toxic: {val_df['label'].mean()*100:.2f}%)")
print(f"Test set  : {len(test_df):,} baris (Toxic: {test_df['label'].mean()*100:.2f}%)")
print(f"Lokasi file split tersimpan: {saved_paths}")

### 4. Pembentukan Vocabulary & Integer Sequences (`TextTokenizer`)
**Aturan Reprodusibilitas & Integritas:** `tokenizer.fit()` dijalankan secara eksklusif hanya pada `train_df['text_clean']`.

In [ ]:
tokenizer = TextTokenizer(
    vocab_size=Config.VOCAB_SIZE,
    oov_token=Config.OOV_TOKEN,
    pad_token=Config.PAD_TOKEN
)

# Fit HANYA pada data train
print("Fitting tokenizer pada Train corpus...")
tokenizer.fit(train_df["text_clean"].tolist())
print(f"Ukuran vocabulary yang terindeks: {len(tokenizer.word_index):,} kata")

# Transform ke integer sequence IDs
train_seqs = tokenizer.texts_to_sequences(train_df["text_clean"].tolist())
val_seqs = tokenizer.texts_to_sequences(val_df["text_clean"].tolist())
test_seqs = tokenizer.texts_to_sequences(test_df["text_clean"].tolist())

padder = SequencePadder(max_len=Config.MAX_LEN, padding="post", truncating="post")

X_train = padder.pad(train_seqs)
X_val = padder.pad(val_seqs)
X_test = padder.pad(test_seqs)

print(f"Dimensi Matrix Padded X_train: {X_train.shape}")
print(f"Dimensi Matrix Padded X_val  : {X_val.shape}")
print(f"Dimensi Matrix Padded X_test : {X_test.shape}")

# Simpan tokenizer state untuk deployment UI & testing
tok_save_path = ROOT_DIR / Config.TOKENIZER_OUTPUT
os.makedirs(tok_save_path.parent, exist_ok=True)
tokenizer.save(str(tok_save_path))
print(f"Tokenizer berhasil diserialisasi ke: {tok_save_path}")

### 5. Kesimpulan Milestone M3–M4
1. **Dataset Skala Nyata**: Seluruh 28.448 baris telah dibersihkan secara konsisten dan terfilter dari noise.
2. **Partisi Terisolasi**: Pembagian 70/15/15 (train/val/test) tersimpan di `data/splits/` dengan stratified balance teruji.
3. **Bebas Data Leakage**: Tokenizer hanya belajar dari distribusi korpus train dan siap disuplai ke model baseline (M5) dan CNN (M6-M7).